In [1]:
!pip install ultralytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.2/41.2 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 20.5 MB/s eta 0:00:00


In [13]:
import kagglehub
import os
from sklearn.model_selection import train_test_split
from pathlib import Path
import shutil
import yaml
import torch
from ultralytics import YOLO

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
dataset_path = kagglehub.dataset_download('mahmoudeldebase/egyptian-cars-plates')
print("Dataset path:", dataset_path)

100%|██████████| 196M/196M [00:01<00:00, 124MB/s]

Extracting files...


Dataset path: /root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles Labeling/1967.txt
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles Labeling/0408.txt
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles Labeling/0491.txt
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles Labeling/0396.txt
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles Labeling/1898.txt
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles/1384.jpg
/root/.cache/kagglehub/datasets/mahmoudeldebase/egyptian-cars-plates/versions/1/EALPR Vechicles dataset/Vehicles/1436.jpg
/root/.

In [7]:
dataset_path_base = os.path.join(dataset_path, 'EALPR Vechicles dataset')
image_dir = os.path.join(dataset_path_base, 'Vehicles')
label_dir = os.path.join(dataset_path_base, 'Vehicles Labeling')

images = sorted([f for f in os.listdir(image_dir) if f.endswith('.jpg')])
labels = sorted([f for f in os.listdir(label_dir) if f.endswith('.txt')])

print(f"Images: {len(images)}")
print(f"Labels: {len(labels)}")

# Check all images have a matching label
matched = [f for f in images if f.replace('.jpg', '.txt') in labels]
print(f"Matched pairs: {len(matched)}")

Images: 2031
Labels: 2088
Matched pairs: 2031


In [9]:
train_imgs, val_imgs = train_test_split(matched, test_size=0.15, random_state=42)

for split, imgs in [('train', train_imgs), ('valid', val_imgs)]:
    Path(f'/content/{split}/images').mkdir(parents=True, exist_ok=True)
    Path(f'/content/{split}/labels').mkdir(parents=True, exist_ok=True)
    for fname in imgs:
        shutil.copy(os.path.join(image_dir, fname),
                    f'/content/{split}/images/{fname}')
        shutil.copy(os.path.join(label_dir, fname.replace('.jpg', '.txt')),
                    f'/content/{split}/labels/{fname.replace(".jpg", ".txt")}')

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

Train: 1726 | Val: 305


In [10]:
for split in ['train', 'valid']:
    imgs = len(os.listdir(f'/content/{split}/images'))
    lbls = len(os.listdir(f'/content/{split}/labels'))
    print(f"{split}: {imgs} images, {lbls} labels")


train: 1726 images, 1726 labels
valid: 305 images, 305 labels


In [12]:
data_config = {
    'train': '/content/train/images',
    'val':   '/content/valid/images',
    'nc':    1,
    'names': ['plate']
}

with open('plate_data.yaml', 'w') as f:
    yaml.dump(data_config, f)

print("plate_data.yaml created")

plate_data.yaml created


In [14]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = YOLO('yolo11n.pt')

model.train(
    data='plate_data.yaml',
    epochs=100,
    batch=16,
    imgsz=640,
    project='/content/drive/MyDrive/YOLO11-EgyptPlate',
    name='exp1',
    device=device,
    exist_ok=True,
    patience=15,
    optimizer='SGD',
    lr0=0.001,
    lrf=0.01,
)

print("Training complete!")

Using device: cuda
Ultralytics 8.4.60 🚀 Python-3.12.13 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=plate_data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=exp1, nbs=64, nms=False, opset=None, optimize=False, optimizer=SGD, overlap_mask=True, patience=15,